# DenseNet-121 trên CIFAR-10
## Mục tiêu
- Sử dụng DenseNet-121 pretrained cho CIFAR-10.
- Thay classifier từ 1000 → 10 lớp.
- Thực hiện Transfer Learning và Fine-tuning.
- Huấn luyện với Batch Size 32 và 64.
- So sánh kết quả và đưa ra kết luận.
## 1. Dữ liệu CIFAR-10
Sử dụng DataLoader chung của nhóm, ảnh được tiền xử lý về kích thước 224×224 để phù hợp với DenseNet-121 pretrained.

In [29]:
from pathlib import Path
import sys
import time
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim


# ============================================================
# Xác định thư mục Practice_2
# ============================================================

PROJECT_DIR = Path.cwd().resolve()

while (
    PROJECT_DIR.name != "Practice_2"
    and PROJECT_DIR.parent != PROJECT_DIR
):
    PROJECT_DIR = PROJECT_DIR.parent

if PROJECT_DIR.name != "Practice_2":
    raise FileNotFoundError("Không tìm thấy thư mục Practice_2.")

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))


# ============================================================
# Module dùng chung của nhóm
# ============================================================

from src.data import (
    CIFAR10_CLASSES,
    IMAGENET_MEAN,
    IMAGENET_STD,
    DataConfig,
    build_dataloaders,
)

from src.trainer import (
    get_device,
    train_model,
)

# ============================================================
# Module DenseNet-121 
# ============================================================

from src.models.densenet121 import (
    build_densenet121,
    enable_finetuning_last_block,
    count_parameters,
)

print("Import thành công!")
print("PROJECT_DIR:", PROJECT_DIR)
# ============================================================
# THIẾT LẬP RANDOM SEED VÀ DEVICE
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# Sử dụng hàm get_device() từ trainer.py của nhóm
device = get_device()

print("===== CẤU HÌNH MÔI TRƯỜNG =====")
print(f"Random Seed     : {SEED}")
print(f"PyTorch Version : {torch.__version__}")
print(f"Device          : {device}")
print(f"CUDA Available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")

# ============================================================
# KIỂM TRA DATALOADER CIFAR-10 DÙNG CHUNG
# ============================================================

data_config = DataConfig(
    data_dir=PROJECT_DIR / "data",
    image_size=224,
    batch_size=32,
    val_ratio=0.10,
    num_workers=2,
    seed=SEED,
)

data_bundle = build_dataloaders(data_config)

# Dataset
train_dataset = data_bundle["train_dataset"]
val_dataset = data_bundle["val_dataset"]
test_dataset = data_bundle["test_dataset"]

# DataLoader
train_loader = data_bundle["train_loader"]
val_loader = data_bundle["val_loader"]
test_loader = data_bundle["test_loader"]

# Tên các lớp
class_names = data_bundle["class_names"]


print("===== THÔNG TIN CIFAR-10 =====")
print(f"Train samples      : {len(train_dataset):,}")
print(f"Validation samples : {len(val_dataset):,}")
print(f"Test samples       : {len(test_dataset):,}")
print(f"Number of classes  : {len(class_names)}")
print(f"Classes            : {class_names}")


# ============================================================
# KIỂM TRA MỘT BATCH
# ============================================================

images, labels = next(iter(train_loader))

print("\n===== KIỂM TRA MỘT BATCH =====")
print(f"Images shape : {images.shape}")
print(f"Labels shape : {labels.shape}")
print(f"Image dtype  : {images.dtype}")
print(f"Label dtype  : {labels.dtype}")

Import thành công!
PROJECT_DIR: C:\Users\LAM LINH\OneDrive\Tài liệu\TQHDL\UTH-Deep-Learning-nhom2\Practice_2
===== CẤU HÌNH MÔI TRƯỜNG =====
Random Seed     : 42
PyTorch Version : 2.9.0+cpu
Device          : cpu
CUDA Available  : False
===== THÔNG TIN CIFAR-10 =====
Train samples      : 45,000
Validation samples : 5,000
Test samples       : 10,000
Number of classes  : 10
Classes            : ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

===== KIỂM TRA MỘT BATCH =====
Images shape : torch.Size([32, 3, 224, 224])
Labels shape : torch.Size([32])
Image dtype  : torch.float32
Label dtype  : torch.int64


## 2. DenseNet-121 Pretrained và Classifier
Sử dụng DenseNet-121 pretrained trên ImageNet và thay `model.classifier` từ 1000 lớp thành 10 lớp cho CIFAR-10.

Kiểm tra kiến trúc, số tham số và Forward Pass để xác nhận mô hình tương thích với dữ liệu đầu vào `[B, 3, 224, 224]` và tạo output `[B, 10]`.

In [30]:
from torchvision import models

# ============================================================
# DenseNet-121 gốc pre-trained trên ImageNet
# ===========================================================

weights = models.DenseNet121_Weights.DEFAULT

original_model = models.densenet121(
    weights=weights
)

original_classifier = original_model.classifier


# ===========================================================
# DenseNet-121 được điều chỉnh cho CIFAR-10
# Sử dụng module densenet121.py
# ===========================================================

model = build_densenet121(
    num_classes=10,
    pretrained=True,
    freeze_backbone=True
)

adapted_classifier = model.classifier

# ===========================================================
# So sánh classifier trước và sau khi điều chỉnh
# ===========================================================

print("===== DENSENET-121 PRE-TRAINED =====")

print("\nClassifier gốc (ImageNet):")
print(original_classifier)

print("\nClassifier sau khi điều chỉnh cho CIFAR-10:")
print(adapted_classifier)

print("\n===== SO SÁNH =====")
print(
    f"Input features       : "
    f"{adapted_classifier.in_features}"
)

print(
    f"Output gốc ImageNet  : "
    f"{original_classifier.out_features}"
)

print(
    f"Output mới CIFAR-10  : "
    f"{adapted_classifier.out_features}"
)


# ===========================================================
# Khảo sát
# ===========================================================

assert original_classifier.out_features == 1000
assert adapted_classifier.out_features == 10

print("\nKiểm tra thành công: classifier đã được thay từ 1000 → 10 lớp.")

print("===== CẤU TRÚC CHÍNH CỦA DENSENET-121 =====")

for name, module in model.features.named_children():
    print(f"{name:<12} -> {module.__class__.__name__}")

print("\nClassifier:")
print(model.classifier)


# ============================================================
# THỐNG KÊ PARAMETERS
# ============================================================

total_params, trainable_params = count_parameters(model)

frozen_params = total_params - trainable_params

print("\n===== THỐNG KÊ PARAMETERS =====")
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
print(f"Frozen parameters    : {frozen_params:,}")

print(
    f"Trainable ratio      : "
    f"{trainable_params / total_params * 100:.4f}%"
)
# ============================================================
# FORWARD PASS VỚI MỘT BATCH CIFAR-10
# ============================================================

model = model.to(device)

# Sử dụng batch CIFAR-10 đã lấy từ train_loader
input_images = images.to(device)
input_labels = labels.to(device)

model.eval()

with torch.no_grad():
    logits = model(input_images)

print("===== FORWARD PASS =====")
print(f"Input shape        : {input_images.shape}")
print(f"Output shape       : {logits.shape}")
print(f"Output dtype       : {logits.dtype}")
print(f"Number of classes  : {logits.shape[1]}")


# ============================================================
# KIỂM TRA 
# ============================================================

assert input_images.shape == (32, 3, 224, 224), \
    "Kích thước input không đúng."

assert logits.shape == (32, 10), \
    "Output của DenseNet-121 phải có 10 lớp."

assert torch.isfinite(logits).all(), \
    "Output chứa NaN hoặc Infinity."


print("\nKiểm tra thành công:")
print("DenseNet-121 nhận đúng dữ liệu CIFAR-10 và trả về 10 logits cho mỗi ảnh.")

===== DENSENET-121 PRE-TRAINED =====

Classifier gốc (ImageNet):
Linear(in_features=1024, out_features=1000, bias=True)

Classifier sau khi điều chỉnh cho CIFAR-10:
Linear(in_features=1024, out_features=10, bias=True)

===== SO SÁNH =====
Input features       : 1024
Output gốc ImageNet  : 1000
Output mới CIFAR-10  : 10

Kiểm tra thành công: classifier đã được thay từ 1000 → 10 lớp.
===== CẤU TRÚC CHÍNH CỦA DENSENET-121 =====
conv0        -> Conv2d
norm0        -> BatchNorm2d
relu0        -> ReLU
pool0        -> MaxPool2d
denseblock1  -> _DenseBlock
transition1  -> _Transition
denseblock2  -> _DenseBlock
transition2  -> _Transition
denseblock3  -> _DenseBlock
transition3  -> _Transition
denseblock4  -> _DenseBlock
norm5        -> BatchNorm2d

Classifier:
Linear(in_features=1024, out_features=10, bias=True)

===== THỐNG KÊ PARAMETERS =====
Total parameters     : 6,964,106
Trainable parameters : 10,250
Frozen parameters    : 6,953,856
Trainable ratio      : 0.1472%
===== FORWARD PASS ====

## 3. Transfer Learning và Fine-tuning

Ở giai đoạn Transfer Learning, toàn bộ backbone được đóng băng và chỉ classifier được huấn luyện.

Sau đó, `denseblock4` và classifier được mở để Fine-tuning, trong khi các layer trước vẫn được giữ cố định.

In [31]:
transfer_total_params, transfer_trainable_params = count_parameters(model)

backbone_frozen = all(
    not p.requires_grad
    for p in model.features.parameters()
)

classifier_trainable = all(
    p.requires_grad
    for p in model.classifier.parameters()
)

print("===== TRANSFER LEARNING =====")
print(f"Total parameters     : {transfer_total_params:,}")
print(f"Trainable parameters : {transfer_trainable_params:,}")
print(f"Backbone frozen      : {backbone_frozen}")
print(f"Classifier trainable : {classifier_trainable}")

assert backbone_frozen is True
assert classifier_trainable is True


# ============================================================
# THIẾT LẬP FINE-TUNING
# ============================================================

model_ft_demo = build_densenet121(
    num_classes=10,
    pretrained=True,
    freeze_backbone=True
)

enable_finetuning_last_block(model_ft_demo)

total_ft_params, finetune_trainable_params = count_parameters(
    model_ft_demo
)

frozen_ft_params = total_ft_params - finetune_trainable_params

# ===========================================================
# Kiểm tra trạng thái các layer
# ===========================================================

conv0_trainable = any(
    p.requires_grad
    for p in model_ft_demo.features.conv0.parameters()
)

denseblock4_trainable = all(
    p.requires_grad
    for p in model_ft_demo.features.denseblock4.parameters()
)

classifier_trainable_ft = all(
    p.requires_grad
    for p in model_ft_demo.classifier.parameters()
)

print("\n===== FINE-TUNING =====")
print(f"Transfer Learning trainable : {transfer_trainable_params:,}")
print(f"Fine-tuning trainable       : {finetune_trainable_params:,}")
print(f"Frozen parameters           : {frozen_ft_params:,}")
print(
    f"Trainable ratio             : "
    f"{finetune_trainable_params / total_ft_params * 100:.2f}%"
)

print("\n===== KIỂM TRA LAYER =====")
print(f"conv0 trainable       : {conv0_trainable}")
print(f"denseblock4 trainable : {denseblock4_trainable}")
print(f"classifier trainable  : {classifier_trainable_ft}")

assert conv0_trainable is False
assert denseblock4_trainable is True
assert classifier_trainable_ft is True

assert finetune_trainable_params > transfer_trainable_params

print(
    "\nKiểm tra thành công: "
    "Transfer Learning và Fine-tuning được thiết lập đúng."
)

===== TRANSFER LEARNING =====
Total parameters     : 6,964,106
Trainable parameters : 10,250
Backbone frozen      : True
Classifier trainable : True

===== FINE-TUNING =====
Transfer Learning trainable : 10,250
Fine-tuning trainable       : 2,168,330
Frozen parameters           : 4,795,776
Trainable ratio             : 31.14%

===== KIỂM TRA LAYER =====
conv0 trainable       : False
denseblock4 trainable : True
classifier trainable  : True

Kiểm tra thành công: Transfer Learning và Fine-tuning được thiết lập đúng.


## 4. Cấu hình thí nghiệm

DenseNet-121 được huấn luyện với hai Batch Size: **32** và **64**.

Để đảm bảo so sánh công bằng, hai thí nghiệm sử dụng cùng Random Seed, Learning Rate, Optimizer, Loss Function, số epoch, Train/Validation split và Data Pipeline. Biến duy nhất thay đổi là Batch Size.

Mỗi thí nghiệm gồm hai giai đoạn:

1. **Transfer Learning:** đóng băng backbone và chỉ huấn luyện classifier.
2. **Fine-tuning:** nạp checkpoint tốt nhất của Transfer Learning, mở `denseblock4` và classifier, sau đó tiếp tục huấn luyện.

In [32]:
# ============================================================
# CẤU HÌNH THÍ NGHIỆM DENSENET-121
# ============================================================

NUM_CLASSES = 10
IMAGE_SIZE = 224

BATCH_SIZE_A = 32
BATCH_SIZE_B = 64

LEARNING_RATE = 0.001
OPTIMIZER_NAME = "Adam"

TRANSFER_EPOCHS = 2
FINETUNE_EPOCHS = 3
TOTAL_EPOCHS = TRANSFER_EPOCHS + FINETUNE_EPOCHS

criterion = nn.CrossEntropyLoss()

CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
RUNS_DIR = PROJECT_DIR / "runs" / "densenet121"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("===== CẤU HÌNH THÍ NGHIỆM =====")
print(f"Batch Size A      : {BATCH_SIZE_A}")
print(f"Batch Size B      : {BATCH_SIZE_B}")
print(f"Optimizer         : {OPTIMIZER_NAME}")
print(f"Learning Rate     : {LEARNING_RATE}")
print(f"Loss Function     : {criterion.__class__.__name__}")
print(f"Transfer Epochs   : {TRANSFER_EPOCHS}")
print(f"Fine-tune Epochs  : {FINETUNE_EPOCHS}")
print(f"Total Epochs      : {TOTAL_EPOCHS}")
print(f"Random Seed       : {SEED}")


# ============================================================
# DATALOADER CHO BATCH SIZE 32 VÀ 64
# ============================================================

# BS32 đã được tạo ở phần dữ liệu
data_bs32 = data_bundle

# Chỉ tạo thêm DataLoader BS64 bằng pipeline chung
config_bs64 = DataConfig(
    data_dir=PROJECT_DIR / "data",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE_B,
    val_ratio=0.10,
    num_workers=2,
    seed=SEED,
)

data_bs64 = build_dataloaders(config_bs64)

train_loader_32 = data_bs32["train_loader"]
val_loader_32 = data_bs32["val_loader"]

train_loader_64 = data_bs64["train_loader"]
val_loader_64 = data_bs64["val_loader"]


# ============================================================
# KIỂM TRA CÙNG TRAIN / VALIDATION SPLIT
# ============================================================

train_same_split = (
    list(data_bs32["train_indices"])
    == list(data_bs64["train_indices"])
)

val_same_split = (
    list(data_bs32["val_indices"])
    == list(data_bs64["val_indices"])
)

print("\n===== SO SÁNH DATALOADER =====")
print(
    f"BS32: Train={len(train_loader_32.dataset):,}, "
    f"Val={len(val_loader_32.dataset):,}, "
    f"Batches={len(train_loader_32)}"
)

print(
    f"BS64: Train={len(train_loader_64.dataset):,}, "
    f"Val={len(val_loader_64.dataset):,}, "
    f"Batches={len(train_loader_64)}"
)

print(f"Cùng Train split      : {train_same_split}")
print(f"Cùng Validation split : {val_same_split}")

assert len(train_loader_32.dataset) == 45000
assert len(train_loader_64.dataset) == 45000
assert len(val_loader_32.dataset) == 5000
assert len(val_loader_64.dataset) == 5000

assert train_same_split
assert val_same_split

print(
    "\nKiểm tra thành công: "
    "hai thí nghiệm chỉ khác Batch Size."
)

===== CẤU HÌNH THÍ NGHIỆM =====
Batch Size A      : 32
Batch Size B      : 64
Optimizer         : Adam
Learning Rate     : 0.001
Loss Function     : CrossEntropyLoss
Transfer Epochs   : 2
Fine-tune Epochs  : 3
Total Epochs      : 5
Random Seed       : 42

===== SO SÁNH DATALOADER =====
BS32: Train=45,000, Val=5,000, Batches=1407
BS64: Train=45,000, Val=5,000, Batches=704
Cùng Train split      : True
Cùng Validation split : True

Kiểm tra thành công: hai thí nghiệm chỉ khác Batch Size.


## 5. Quy trình huấn luyện hai giai đoạn

Mỗi thí nghiệm sử dụng `train_model()` từ training engine chung của nhóm.

Ở giai đoạn Transfer Learning, chỉ classifier được tối ưu và checkpoint có Validation Accuracy tốt nhất được lưu lại.

Checkpoint tốt nhất sau đó được nạp bằng `map_location=device`. `denseblock4` và classifier được mở để Fine-tuning, đồng thời một optimizer mới được khởi tạo để cập nhật các tham số vừa được mở.

In [33]:
import shutil


# ============================================================
# HỖ TRỢ NẠP CHECKPOINT
# ============================================================

def load_checkpoint_to_model(model, checkpoint_path, device):
    checkpoint = torch.load(
        checkpoint_path,
        map_location=device
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    return checkpoint


# ============================================================
# CHẠY MỘT THÍ NGHIỆM DENSENET-121
# ============================================================

def run_densenet_experiment(
    batch_size,
    train_loader,
    val_loader,
):
    print("=" * 65)
    print(
        f"DENSENET-121 EXPERIMENT | "
        f"BATCH SIZE = {batch_size}"
    )
    print("=" * 65)

    # ============================================================
    # Reset seed để hai experiment công bằng
    # ============================================================

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    # ============================================================
    # Tạo model mới
    # ============================================================

    experiment_model = build_densenet121(
        num_classes=NUM_CLASSES,
        pretrained=True,
        freeze_backbone=True,
    )

    experiment_name = f"densenet121_bs{batch_size}"

    transfer_log_dir = (
        RUNS_DIR / experiment_name / "transfer"
    )

    finetune_log_dir = (
        RUNS_DIR / experiment_name / "finetune"
    )

    transfer_checkpoint = (
        CHECKPOINT_DIR
        / f"{experiment_name}_transfer_best.pth"
    )

    finetune_checkpoint = (
        CHECKPOINT_DIR
        / f"{experiment_name}_finetune_best.pth"
    )

    final_checkpoint = (
        CHECKPOINT_DIR
        / f"{experiment_name}_best.pth"
    )

    # ============================================================
    # Đo thời gian
    # ============================================================

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    start_time = time.perf_counter()


    # ========================================================
    # STAGE 1 — TRANSFER LEARNING
    # ========================================================

    print("\n[STAGE 1] TRANSFER LEARNING")

    _, transfer_params = count_parameters(
        experiment_model
    )

    print(
        f"Trainable parameters: "
        f"{transfer_params:,}"
    )

    optimizer_transfer = optim.Adam(
        filter(
            lambda p: p.requires_grad,
            experiment_model.parameters()
        ),
        lr=LEARNING_RATE,
    )

    history_transfer = train_model(
        model=experiment_model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer_transfer,
        num_epochs=TRANSFER_EPOCHS,
        log_dir=str(transfer_log_dir),
        checkpoint_path=str(transfer_checkpoint),
    )


    # ========================================================
    # NẠP BEST TRANSFER CHECKPOINT
    # ========================================================

    transfer_ckpt = load_checkpoint_to_model(
        experiment_model,
        transfer_checkpoint,
        device,
    )

    transfer_best_acc = transfer_ckpt["val_acc"]

    print(
        f"\nBest Transfer Val Acc: "
        f"{transfer_best_acc:.2f}%"
    )


    # ========================================================
    # STAGE 2 — FINE-TUNING
    # ========================================================

    print("\n[STAGE 2] FINE-TUNING")

    enable_finetuning_last_block(
        experiment_model
    )

    _, finetune_params = count_parameters(
        experiment_model
    )

    print(
        f"Trainable parameters: "
        f"{finetune_params:,}"
    )

    # Bắt buộc tạo optimizer mới sau khi mở denseblock4
    optimizer_finetune = optim.Adam(
        filter(
            lambda p: p.requires_grad,
            experiment_model.parameters()
        ),
        lr=LEARNING_RATE,
    )

    history_finetune = train_model(
        model=experiment_model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer_finetune,
        num_epochs=FINETUNE_EPOCHS,
        log_dir=str(finetune_log_dir),
        checkpoint_path=str(finetune_checkpoint),
    )


    # ========================================================
    # BEST FINE-TUNING CHECKPOINT
    # ========================================================

    finetune_ckpt = torch.load(
        finetune_checkpoint,
        map_location=device,
    )

    finetune_best_acc = finetune_ckpt["val_acc"]


    # ========================================================
    # CHỌN BEST CHECKPOINT TOÀN BỘ
    # ========================================================

    if finetune_best_acc >= transfer_best_acc:
        best_stage = "Fine-tuning"
        best_val_acc = finetune_best_acc
        best_source = finetune_checkpoint
    else:
        best_stage = "Transfer Learning"
        best_val_acc = transfer_best_acc
        best_source = transfer_checkpoint

    shutil.copy2(
        best_source,
        final_checkpoint
    )


    # ========================================================
    # THỜI GIAN VÀ GPU MEMORY
    # ========================================================

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    training_time = (
        time.perf_counter() - start_time
    )

    if torch.cuda.is_available():
        peak_memory_mb = (
            torch.cuda.max_memory_allocated()
            / (1024 ** 2)
        )
    else:
        peak_memory_mb = None


    # ========================================================
    # GỘP HISTORY
    # ========================================================

    history = {
        "train_loss":
            history_transfer["train_loss"]
            + history_finetune["train_loss"],

        "train_acc":
            history_transfer["train_acc"]
            + history_finetune["train_acc"],

        "val_loss":
            history_transfer["val_loss"]
            + history_finetune["val_loss"],

        "val_acc":
            history_transfer["val_acc"]
            + history_finetune["val_acc"],
    }


    # ========================================================
    # KẾT QUẢ
    # ========================================================

    return {
        "batch_size": batch_size,
        "history": history,
        "history_transfer": history_transfer,
        "history_finetune": history_finetune,
        "transfer_best_acc": transfer_best_acc,
        "finetune_best_acc": finetune_best_acc,
        "best_val_acc": best_val_acc,
        "best_stage": best_stage,
        "training_time": training_time,
        "peak_memory_mb": peak_memory_mb,
        "checkpoint": str(final_checkpoint),
    }


print("===== TRAINING PIPELINE =====")
print("run_densenet_experiment:", callable(run_densenet_experiment))
print("Pipeline đã sẵn sàng.")

===== TRAINING PIPELINE =====
run_densenet_experiment: True
Pipeline đã sẵn sàng.


## 6. Thí nghiệm A — DenseNet-121 với Batch Size = 32

Thí nghiệm đầu tiên huấn luyện DenseNet-121 với **Batch Size = 32** theo cấu hình chung đã thiết lập.

Quá trình huấn luyện gồm hai giai đoạn:

1. **Transfer Learning:** đóng băng backbone và chỉ huấn luyện classifier trong 2 epochs.
2. **Fine-tuning:** nạp checkpoint tốt nhất của Transfer Learning, mở `denseblock4` và classifier, sau đó tiếp tục huấn luyện trong 3 epochs.

Toàn bộ quá trình sử dụng `train_model()` từ training engine chung của nhóm.

In [34]:
# ============================================================
# EXPERIMENT A — BATCH SIZE = 32
# ============================================================

result_bs32 = run_densenet_experiment(
    batch_size=BATCH_SIZE_A,
    train_loader=train_loader_32,
    val_loader=val_loader_32,
)

DENSENET-121 EXPERIMENT | BATCH SIZE = 32

[STAGE 1] TRANSFER LEARNING
Trainable parameters: 10,250

Epoch [1/2]


Training:   3%|▎         | 36/1407 [00:59<37:42,  1.65s/it]


KeyboardInterrupt: 

In [ ]:
# ============================================================
# TỔNG HỢP KẾT QUẢ — BATCH SIZE 32
# ============================================================

import json

history_32 = result_bs32["history"]

finetune_gain_bs32 = (
    result_bs32["finetune_best_acc"]
    - result_bs32["transfer_best_acc"]
)

training_hours_bs32 = (
    result_bs32["training_time"] / 3600
)

print("===== KẾT QUẢ BATCH SIZE = 32 =====")

print(
    f"Best Transfer Val Acc : "
    f"{result_bs32['transfer_best_acc']:.2f}%"
)

print(
    f"Best Fine-tuning Acc  : "
    f"{result_bs32['finetune_best_acc']:.2f}%"
)

print(
    f"Fine-tuning Gain      : "
    f"{finetune_gain_bs32:.2f} điểm %"
)

print(
    f"Best Val Accuracy     : "
    f"{result_bs32['best_val_acc']:.2f}%"
)

print(
    f"Best Stage            : "
    f"{result_bs32['best_stage']}"
)

print(
    f"Training Time         : "
    f"{result_bs32['training_time']:.2f} giây "
    f"({training_hours_bs32:.2f} giờ)"
)

if result_bs32["peak_memory_mb"] is not None:
    print(
        f"Peak GPU Memory       : "
        f"{result_bs32['peak_memory_mb']:.2f} MB"
    )
else:
    print("Peak GPU Memory       : Không ghi nhận (CPU)")

print(
    f"Best Checkpoint       : "
    f"{result_bs32['checkpoint']}"
)


# ============================================================
# LƯU KẾT QUẢ JSON
# ============================================================

result_path_bs32 = (
    PROJECT_DIR
    / "results"
    / "densenet121_bs32_results.json"
)

result_to_save = {
    "model": "DenseNet-121",
    "batch_size": result_bs32["batch_size"],
    "optimizer": OPTIMIZER_NAME,
    "learning_rate": LEARNING_RATE,
    "transfer_epochs": TRANSFER_EPOCHS,
    "finetune_epochs": FINETUNE_EPOCHS,
    "history": result_bs32["history"],
    "history_transfer": result_bs32["history_transfer"],
    "history_finetune": result_bs32["history_finetune"],
    "transfer_best_acc": result_bs32["transfer_best_acc"],
    "finetune_best_acc": result_bs32["finetune_best_acc"],
    "best_val_acc": result_bs32["best_val_acc"],
    "best_stage": result_bs32["best_stage"],
    "training_time": result_bs32["training_time"],
    "peak_memory_mb": result_bs32["peak_memory_mb"],
    "checkpoint": result_bs32["checkpoint"],
}

with open(
    result_path_bs32,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        result_to_save,
        f,
        indent=4,
        ensure_ascii=False
    )

best_checkpoint_bs32 = Path(
    result_bs32["checkpoint"]
)

assert result_path_bs32.exists()
assert best_checkpoint_bs32.exists()

print("\n===== ARTIFACTS BS32 =====")
print("JSON       :", result_path_bs32)
print("Checkpoint :", best_checkpoint_bs32)

NameError: name 'result_bs32' is not defined

### Nhận xét Batch Size = 32

Ở giai đoạn Transfer Learning, Best Validation Accuracy đạt **82.56%**. Sau khi mở `denseblock4` để Fine-tuning, Best Validation Accuracy tăng lên **91.38%**, cải thiện **8.82 điểm phần trăm**.

Validation Loss giảm từ **0.5465** ở cuối Transfer Learning xuống **0.2601** ở cuối Fine-tuning. Trong 5 epoch thực nghiệm, Train/Validation Loss đều giảm và Accuracy tiếp tục tăng, chưa xuất hiện dấu hiệu overfitting rõ ràng.

Checkpoint tốt nhất được tạo ở giai đoạn **Fine-tuning**, với Validation Accuracy **91.38%**. Tổng thời gian huấn luyện trên CPU khoảng **27,686.89 giây (~7 giờ 41 phút)**.

### 5.2. Learning Curves của Batch Size = 32

Train Loss, Validation Loss, Train Accuracy và Validation Accuracy
được biểu diễn theo từng epoch.

Đường nét đứt đánh dấu thời điểm chuyển từ:

**Transfer Learning → Fine-tuning**

Qua đó có thể đánh giá Fine-tuning có giúp cải thiện khả năng học
và Validation Accuracy của DenseNet-121 hay không.

In [ ]:
# ============================================================
# LOSS CURVES — BATCH SIZE 32
# ============================================================

plt.figure(figsize=(9, 5))

plt.plot(
    epochs,
    history_32["train_loss"],
    marker="o",
    label="Train Loss"
)

plt.plot(
    epochs,
    history_32["val_loss"],
    marker="o",
    label="Validation Loss"
)

# Vị trí bắt đầu Fine-tuning
plt.axvline(
    x=TRANSFER_EPOCHS + 0.5,
    linestyle="--",
    label="Start Fine-tuning"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title(
    "DenseNet-121 Loss Curves - Batch Size 32"
)

plt.xticks(list(epochs))
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()


loss_figure_bs32 = (
    PROJECT_DIR
    / "results"
    / "figures"
    / "densenet121_bs32_loss.png"
)

plt.savefig(
    loss_figure_bs32,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Đã lưu biểu đồ:")
print(loss_figure_bs32)
# ============================================================
# ACCURACY CURVES — BATCH SIZE 32
# ============================================================

plt.figure(figsize=(9, 5))

plt.plot(
    epochs,
    history_32["train_acc"],
    marker="o",
    label="Train Accuracy"
)

plt.plot(
    epochs,
    history_32["val_acc"],
    marker="o",
    label="Validation Accuracy"
)

plt.axvline(
    x=TRANSFER_EPOCHS + 0.5,
    linestyle="--",
    label="Start Fine-tuning"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")

plt.title(
    "DenseNet-121 Accuracy Curves - Batch Size 32"
)

plt.xticks(list(epochs))
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()


acc_figure_bs32 = (
    PROJECT_DIR
    / "results"
    / "figures"
    / "densenet121_bs32_accuracy.png"
)

plt.savefig(
    acc_figure_bs32,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Đã lưu biểu đồ:")
print(acc_figure_bs32)

NameError: name 'epochs' is not defined

<Figure size 900x500 with 0 Axes>

# 6. Thí nghiệm B — DenseNet-121 với Batch Size = 64
Thí nghiệm thứ hai sử dụng cùng DenseNet-121 và cùng quy trình huấn luyện
với Experiment A, nhưng thay đổi Batch Size từ **32 lên 64**.

Các điều kiện được giữ nguyên:
- Optimizer: Adam
- Learning Rate: 0.001
- Transfer Learning: 2 epochs
- Fine-tuning: 3 epochs
- Random Seed: 42
- Input Size: 224 × 224
- Train/Validation split không thay đổi

Việc giữ cố định các yếu tố trên giúp đánh giá riêng ảnh hưởng của
Batch Size đến Accuracy, Loss và Training Time.

In [ ]:
# ============================================================
# EXPERIMENT B — BATCH SIZE = 64
# ============================================================

result_bs64 = run_densenet_experiment(
    batch_size=BATCH_SIZE_B,
    train_loader=train_loader_64,
    val_loader=val_loader_64,
)

DENSENET-121 EXPERIMENT | BATCH SIZE = 64

[STAGE 1] TRANSFER LEARNING
---------------------------------------------
Trainable parameters: 10,250

Epoch [1/2]


Validating: 100%|██████████| 79/79 [03:56<00:00,  3.00s/it]


Train Loss: 0.9764, Train Acc: 68.93%
Val Loss: 0.6032, Val Acc: 81.24%
Validation accuracy improved (0.00% --> 81.24%). Saving model...

Epoch [2/2]


Validating: 100%|██████████| 79/79 [03:47<00:00,  2.88s/it]


Train Loss: 0.7191, Train Acc: 75.46%
Val Loss: 0.5625, Val Acc: 81.96%
Validation accuracy improved (81.24% --> 81.96%). Saving model...

Best Transfer Val Acc: 81.96%

[STAGE 2] FINE-TUNING
---------------------------------------------
Trainable parameters: 2,168,330

Epoch [1/3]


Validating:  89%|████████▊ | 70/79 [03:16<00:25,  2.81s/it]


KeyboardInterrupt: 